# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata (this does not download records yet)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
print("Available record sets:")
for rs in metadata.record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, data type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List record set @ids to load (from overview above)
# For this dataset, let's check which record set is the survey data.
# Commonly, survey data is in a main record set with name 'responses' or 'survey'. We'll print all, but pick the main as the example.
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Loaded {len(df)} records.")
    else:
        print("  No records found (possibly documentation/meta record set).")

# For further steps, choose the main record set holding the survey data rows
# We'll select the record set with the most rows as the main survey data for demonstration
if dataframes:
    # Pick the biggest
    survey_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Selected survey record set for analysis: {survey_record_set_id}")
    print("Survey data sample:")
    display(dataframes[survey_record_set_id].head())
else:
    survey_record_set_id = None
    print("No tabular survey record set found. Check metadata record sets above.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping data for further analysis.

In [ ]:
import numpy as np

if survey_record_set_id is not None:
    survey_df = dataframes[survey_record_set_id]

    # Find a numeric field for demonstration. We'll search for fields with float or int values.
    numeric_cols = survey_df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Analyzing numeric field (by @id): {numeric_field}")

        # Remove obvious outliers (> mean + 3 std)
        field_mean = survey_df[numeric_field].mean()
        field_std = survey_df[numeric_field].std()
        upper_limit = field_mean + 3 * field_std

        filtered_df = survey_df[survey_df[numeric_field] < upper_limit]
        print(f"Filtered out records where {numeric_field} >= {upper_limit:.2f}, remaining: {len(filtered_df)}")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - field_mean) / field_std
        print("Sample normalized values:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # If there's a sensible categorical/group field, groupby and aggregate
        # We'll use the first string/object column as group field (not id or uid)
        group_candidates = [col for col in survey_df.select_dtypes(include=[object, 'category']).columns if col not in ['id', 'uid']]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Sample grouped means:")
            display(grouped_df.head())
        else:
            print("No suitable group field found to aggregate.")
    else:
        print("No numeric field found in this record set. Data types are:")
        print(survey_df.dtypes)
else:
    print("No survey dataframe loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if survey_record_set_id is not None and numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(survey_df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If there's a group field, show mean by group
    if group_candidates:
        plt.figure(figsize=(10, 5))
        group_means = survey_df.groupby(group_field)[numeric_field].mean().sort_values()
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
1. Load and inspect metadata and records from a Croissant-metadata AI-ready dataset using `mlcroissant`.
2. Explore record sets, field definitions, and dynamically extract tabular data by referencing entity `@id`s.
3. Perform basic EDA and visualizations on survey fields such as normalizing a numeric field and grouping by demographics.

Further domain-specific analyses can now be applied to discover trends related to mental health, demographics, or regional outcomes using the extracted dataframe. See dataset documentation for details on annotation, consent, and privacy.